<a href="https://colab.research.google.com/github/Mondin0/data-eng/blob/main/CeL_Data_Eng_Realtime_Spark_y_Kafka.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Procesamiento en tiempo real usando Apache Kafka y Apache Spark

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession

In [ ]:
# En entornos diferentes a Databricks
# como un entorno local o Google Colab por ej
# es necesario instanciar una sesion de Spark
# Vamos a instalar tambien el paquete para integrar Kafka y Spark
spark = (SparkSession
         .builder
         .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3")
         .appName("SparkyKafka")
         .getOrCreate()
         )

In [ ]:
# Datos de conexion a Kafka
kafka_usr = "usr"
kafka_pwd = "gpCYTV1Mv0hMZKplGyzWD7uj6O2QBz"

kafka_conf = {
    # Servidor de kafka
    "kafka.bootstrap.servers": "cstoobu6igjosi339b60.any.us-east-1.mpx.prd.cloud.redpanda.com:9092",
    "kafka.group.id": "sparkConsumerClaseUTN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "SCRAM-SHA-256",
    # Configuracion de autenticacion
    "kafka.sasl.jaas.config": f"org.apache.kafka.common.security.scram.ScramLoginModule required username=\"{kafka_usr}\" password=\"{kafka_pwd}\";",
}

## Consumer en Spark
Lectura de datos crudos de Kafka

In [ ]:
# Configuraciones adicionales para un Consumer
kafka_consumer_conf = kafka_conf.copy()
kafka_consumer_conf["subscribe"] = "logs"
kafka_consumer_conf["startingOffsets"] = "earliest"

df_logs = (spark.readStream.format("kafka")
                      .options(**kafka_consumer_conf)
                      .load())

"""df_logs = (spark.read.format("kafka")
                      .options(**kafka_params)
                      .load())"""

'df_logs = (spark.read.format("kafka")\n                      .options(**kafka_params)\n                      .load())'

In [ ]:
# Los datos en si están dentro de un campo value,
# Debemos procesar ese campo, para darles una estructura mas manejable

df_logs_structured = (df_logs
                      .selectExpr(
                        "CAST(value AS STRING) AS raw_data", # Decodificar la columna value
                        "timestamp AS tstamp_received", # Renombrar la estampa de tiempo
                        )
                      .selectExpr(
                        "raw_data", "tstamp_received", # Acceder columnas
                        "CAST(SPLIT(raw_data, ' - ')[0] AS TIMESTAMP) AS tstamp_data",
                        "SPLIT(raw_data, ' - ')[1] AS ip_addr",
                        "SPLIT(raw_data, ' - ')[2] AS http_operation",
                        "CAST(SPLIT(raw_data, ' - ')[3] AS INT) AS status_code",
                        "CAST(SPLIT(raw_data, ' - ')[4] AS INT) AS size",
                      ))

In [ ]:
# Vamos a mostrar "por consola" los resultados de las transformaciones

df_logs_structured_sink = (df_logs_structured.writeStream
 .format("memory")
 .queryName("logs_parsed") # Nombre de tabla temporal, en memoria
 )

df_logs_structured_sink.start()

In [ ]:
spark.sql("select * from logs_parsed").show()

+--------------------+--------------------+-------------------+---------------+--------------------+-----------+-----+
|            raw_data|     tstamp_received|        tstamp_data|        ip_addr|      http_operation|status_code| size|
+--------------------+--------------------+-------------------+---------------+--------------------+-----------+-----+
|ASDSA 8JGGJD  DFG...|2024-11-19 11:37:...|               NULL|           NULL|                NULL|       NULL| NULL|
|2024-11-19 11:42:...|2024-11-19 11:42:...|2024-11-19 11:42:00| 171.143.49.191|   POST main/explore|        403|58554|
|2024-11-19 11:42:...|2024-11-19 11:42:...|2024-11-19 11:42:20| 186.141.31.120|          GET search|        500|55206|
|2024-11-19 11:42:...|2024-11-19 11:42:...|2024-11-19 11:42:08|   15.51.231.50|            GET tags|        404|27098|
|2024-11-19 11:42:...|2024-11-19 11:42:...|2024-11-19 11:42:03|  12.131.36.156|POST categories/w...|        404|15415|
|2024-11-19 11:42:...|2024-11-19 11:42:...|2024-

## Producer en Spark
Ahora Spark se encargará de escribir los eventos en otro tópico de Kafka.

Kafka solo admite dos columnas: key (opcional) y value (obligatorio, que tendrá los datos a enviar)
En este caso, vamos a estructurar los logs como JSON

In [ ]:
# Pre-procesamiento para convertir los logs en JSON

df_logs_structured = df_logs_structured.selectExpr(
                      "CAST(ip_addr AS STRING) AS key",
                      "CAST(TO_JSON(STRUCT(tstamp_data, ip_addr, http_operation, status_code, size)) AS STRING) AS value"
                      )

In [ ]:
# Producer para escribir en Kafka

kafka_producer_conf = kafka_conf.copy()
kafka_producer_conf["topic"] = "logs_parsed"

df_logs_structured_sink = (df_logs_structured.writeStream.format("kafka")
                           .options(**kafka_producer_conf)
                           .option("checkpointLocation", "/dbfs/FileStore/logs_processing")
                           )
df_logs_structured_sink.start()

## Agregaciones

In [ ]:
kafka_consumer_conf = kafka_conf.copy()
kafka_consumer_conf["subscribe"] = "logs_parsed"
kafka_consumer_conf["startingOffsets"] = "earliest"

df_logs_parsed = (spark.readStream.format("kafka")
                      .options(**kafka_consumer_conf)
                      .load())

In [ ]:
# Estructurar el JSON como tabla
log_schema = "struct<tstamp_data:timestamp, ip_addr:string, http_operation:string, status_code:int, size:int>"

df_logs_parsed = (df_logs_parsed
                .selectExpr(f"FROM_JSON(CAST(value AS string), '{log_schema}') AS parsed_record")
                .selectExpr("parsed_record.*")
                )

# Linea para previsualizar los datos en consola
df_logs_parsed.writeStream.format("memory").queryName("logstoagg").start()


In [ ]:
spark.sql("select * from logstoagg").show()

+-------------------+---------------+--------------------+-----------+-----+
|        tstamp_data|        ip_addr|      http_operation|status_code| size|
+-------------------+---------------+--------------------+-----------+-----+
|               NULL|           NULL|                NULL|       NULL| NULL|
|2024-11-19 11:42:00| 171.143.49.191|   POST main/explore|        403|58554|
|2024-11-19 11:42:20| 186.141.31.120|          GET search|        500|55206|
|2024-11-19 11:42:08|   15.51.231.50|            GET tags|        404|27098|
|2024-11-19 11:42:03|  12.131.36.156|POST categories/w...|        404|15415|
|2024-11-19 11:42:31|  15.162.36.218|POST search/wp-co...|        404| 9955|
|2024-11-19 11:42:10| 63.233.202.181|POST list/posts/blog|        403|30339|
|2024-11-19 11:42:17|  107.83.93.122|    GET app/category|        403|25819|
|2024-11-19 11:41:53|102.234.221.252|    GET explore/list|        401|10934|
|2024-11-19 11:42:14| 135.183.185.30| GET tags/wp-content|        401|40666|

In [ ]:
df_logs_parsed = (df_logs_parsed
                  .withColumn("http_method", F.split("http_operation", " ", 2).getItem(0))
                  .withColumn("uri", F.split("http_operation", " ", 2).getItem(1))
                  )

df_logs_windowed_tumbl = (df_logs_parsed
                           .withWatermark("tstamp_data", "30 seconds") # Para manejar retrasos
                           .groupBy(
                             F.window(F.col("tstamp_data"), "1 minutes"), # Ventana de tiempo
                             F.col("http_method")
                             )
                           .count()
                           )

(df_logs_windowed_tumbl.writeStream.format("memory")
 .outputMode("append")
 .queryName("logs_agg_tumb").start()
 )

In [ ]:
spark.sql("select * from logs_agg_tumb").show()

+------+-----------+-----+
|window|http_method|count|
+------+-----------+-----+
+------+-----------+-----+

